# Twin-Demo V2 — PPE 검출 모델 학습 (Colab T4)

YOLO11s · 입력 832 · 100 epoch · 부정형(NO-*) 클래스 강화 설정

**목표:** V1(mAP@50 **0.549**) → V2 개선치 측정

---
### 실행 전 준비 (한 번만)
1. 상단 메뉴 **런타임 → 런타임 유형 변경 → 하드웨어 가속기: T4 GPU** 선택 후 저장
2. 데이터셋용 **무료 Roboflow API 키** 준비:
   - https://roboflow.com 가입(무료) → 우상단 프로필 → **Settings → API Keys → Private API Key** 복사

그다음 셀을 **위에서부터 ▶ 차례로** 실행하세요. (전체 약 1.5~2.5시간, 조기종료 가능)


### 1. GPU 확인


In [ ]:
import torch
ok = torch.cuda.is_available()
print('GPU 사용 가능:', ok, '|', torch.cuda.get_device_name(0) if ok else 'GPU 없음 → 런타임 유형을 T4로 변경하세요')


### 2. 라이브러리 설치 (약 1분)


In [ ]:
!pip -q install ultralytics roboflow


### 3. 데이터셋 받기
아래 따옴표 안에 본인 **Roboflow API 키**를 붙여넣고 실행하세요.


In [ ]:
ROBOFLOW_API_KEY = ""   # ← 여기에 키 붙여넣기

assert ROBOFLOW_API_KEY, '키를 먼저 붙여넣으세요'
from roboflow import Roboflow
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
proj = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
ds = proj.version(27).download('yolov11', location='/content/construction-ppe', overwrite=True)
DATA_YAML = ds.location + '/data.yaml'
print('데이터셋 준비 완료:', DATA_YAML)


### 4. V2 학습
yolo11s · 832 · AdamW + cosine LR · augmentation 강화 · cls 가중치 0.7(부정형 강조).
진행 막대가 epoch마다 갱신됩니다. 메모리 부족(OOM) 시 `batch=16`을 `12` 또는 `8`로 줄이세요.


In [ ]:
import os
os.environ['WANDB_MODE'] = 'disabled'
from ultralytics import YOLO, settings
settings.update({'wandb': False})

model = YOLO('yolo11s.pt')
model.train(
    data=DATA_YAML,
    epochs=100, imgsz=832, batch=16, device=0,
    optimizer='AdamW', lr0=0.002, lrf=0.01, cos_lr=True,
    warmup_epochs=5, patience=15,
    mosaic=1.0, mixup=0.15, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    fliplr=0.5, scale=0.5, degrees=5.0,
    box=7.5, cls=0.7, dfl=1.5,
    project='runs', name='exp02_ppe_yolo11s_v2', exist_ok=True,
    save=True, plots=True,
)


### 5. V2 성능 측정 + V1 비교


In [ ]:
best = 'runs/exp02_ppe_yolo11s_v2/weights/best.pt'
r = YOLO(best).val(data=DATA_YAML, imgsz=832, split='val')
print('='*32)
print('V2 mAP@50    :', round(r.box.map50, 4))
print('V2 mAP@50-95 :', round(r.box.map, 4))
print('-'*32)
for i, ap in enumerate(r.box.ap50):
    print(f'  {r.names[i]:16s} {ap:.4f}')
print('='*32)
print('V1 0.549  →  V2', round(r.box.map50, 4), f'({(r.box.map50-0.549)*100:+.1f}p)')


### 6. 결과 내려받기
`best.pt` + 학습 곡선·혼동행렬·results.csv 가 zip으로 다운로드됩니다.
이 zip을 채팅에 올려주시면 포트폴리오에 반영합니다.


In [ ]:
import shutil
shutil.make_archive('/content/twin_v2_result', 'zip', 'runs/exp02_ppe_yolo11s_v2')
from google.colab import files
files.download('/content/twin_v2_result.zip')
